# BTP: explainable AI-content detection, Kaggle GPU runner
Settings: **Accelerator = GPU T4**, **Internet = On**. Then *Save Version -> Save & Run All (Commit)* so it runs in the background. Outputs (models, features, metrics, logs) appear under *Output*.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /kaggle/working
!rm -rf BTP-1 && git clone -q --depth 1 https://github.com/lakshayg2005/BTP-1.git
%cd BTP-1
# big, re-downloadable things go to /kaggle/tmp so the saved output stays small
!mkdir -p /kaggle/tmp/data /kaggle/tmp/hf && ln -sfn /kaggle/tmp/data data
import os
os.environ.update(PY='python', HF_HOME='/kaggle/tmp/hf', TMPDIR_BTP='/kaggle/tmp/t',
                  IMG_TRAIN='3000', IMG_TEST='500', SID_TRAIN='1500', SID_TEST='500', TXT_TRAIN='2000', TXT_TEST='400', BTP_TEXT_LM_SIZE='1.5B')

In [ ]:
!bash scripts/run_phase.sh deps 2>&1 | tail -3
!python -c "import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.is_available())"

## Phase 1: image AI detector

In [ ]:
!bash scripts/run_phase.sh image 2>&1 | grep -v -E 'it/s\]|s/it\]' | tail -60

## Phase 2: edit localizer (SID-Set masks + synthetic edits)

In [ ]:
!bash scripts/run_phase.sh tamper 2>&1 | grep -v -E 'it/s\]|s/it\]' | tail -30

## Phase 3: audio

In [ ]:
!bash scripts/run_phase.sh audio 2>&1 | grep -v -E 'it/s\]|s/it\]' | tail -60

## Phase 4: text

In [ ]:
!bash scripts/run_phase.sh text 2>&1 | grep -v -E 'it/s\]|s/it\]' | tail -60

## Evaluation: results tables + example explanations

In [ ]:
!bash scripts/run_phase.sh evaluate 2>&1 | grep -v -E 'it/s\]|s/it\]' | tail -150

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('results/summary.md').read()))

In [ ]:
# keep small outputs only; big data stays in /kaggle/tmp
!cd /kaggle/working && zip -qr btp_outputs.zip BTP-1/models BTP-1/results BTP-1/logs BTP-1/features -x '*.npz' && ls -la btp_outputs.zip

## Live demo (interactive sessions only)
Open the notebook in edit mode, run all cells, then this one prints a public gradio link.

In [ ]:
import os
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE') == 'Interactive':
    !python -m app.demo --share
else:
    print('Batch run: demo skipped')